In [1]:
import hail as hl
import os

from ukb_utils import hail_init
from ukb_utils import genotypes
from ko_utils import qc
from ko_utils import analysis

In [2]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_test_export.log', 'GRCh38')

Running on Apache Spark version 3.1.2
SparkUI available at http://compe003.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.77-684f32d73643
LOGGING: writing to logs/hail/hail_test_export.log


In [160]:
path1 = 'data/mt/ukb_wes_200k_annotated_chr21.mt'
mt1 = hl.read_matrix_table(path1)

In [161]:
# load VEP annotation table
input_annotation_path = "data/vep/hail/ukb_wes_200k_chr21_vep.ht"
consequence_annotations = hl.read_table(input_annotation_path)
mt1 = mt1.annotate_rows(consequence = consequence_annotations[mt1.row_key]) 

In [162]:
# annotate consequence
#mt1 = mt1.explode_rows(mt1.consequence.vep.worst_csq_by_gene_canonical)
#by_gene_annotation1 = analysis.annotation_case_builder(mt1.consequence.vep.worst_csq_by_gene_canonical, mt1.consequence.dbnsfp, use_loftee = False)
#mt1 = mt1.annotate_rows(consequence_category = by_gene_annotation1) 
by_gene_annotation = analysis.annotation_case_builder(mt1.consequence.vep.worst_csq_for_variant_canonical, mt1.consequence.dbnsfp, use_loftee = False)
mt1 = mt1.annotate_rows(consequence_category = by_gene_annotation) 

In [165]:
mt1 = count_urv_by_samples(mt1)
mt1 = count_homozygous_urv_by_samples(mt1)
mt1 = count_urv_by_gene(mt1)

In [166]:
mt1.entries().flatten().describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'gene_id': str 
    's': str 
    'eur': int32 
    'n_coding_URV_SNP': int64 
    'n_coding_URV_indel': int64 
    'n_URV_PTV': int64 
    'n_URV_PTV_LC': int64 
    'n_URV_damaging_missense': int64 
    'n_URV_other_missense': int64 
    'n_URV_synonymous': int64 
    'n_URV_non_coding': int64 
    'n_hom_coding_URV_SNP': int64 
    'n_hom_coding_URV_indel': int64 
    'n_hom_URV_PTV': int64 
    'n_hom_URV_PTV_LC': int64 
    'n_hom_URV_damaging_missense': int64 
    'n_hom_URV_other_missense': int64 
    'n_hom_URV_synonymous': int64 
    'n_hom_URV_non_coding': int64 
    'by_gene.coding_URV_SNP': int64 
    'by_gene.coding_URV_indel': int64 
    'by_gene.URV_PTV': int64 
    'by_gene.URV_PTV_LC': int64 
    'by_gene.URV_damaging_missense': int64 
    'by_gene.URV_other_missense': int64 
    'by_gene.URV_synonymous': int64 
    'by_gene.URV_non_coding': int64 


2021-10-25 13:14:56 Hail: INFO: Ordering unsorted dataset with network shuffle


In [154]:
def count_urv_by_samples(mt):
    r'''Count up ultra rare variants by cols and cateogry
    :param mt: a MatrixTable with the field "consequence_category"
    '''
    return mt.annotate_cols(n_coding_URV_SNP = hl.agg.count_where(mt.GT.is_non_ref() & hl.is_snp(mt.alleles[0], mt.alleles[1]) & (mt.consequence_category != "non_coding")),
                          n_coding_URV_indel = hl.agg.count_where(mt.GT.is_non_ref() & hl.is_indel(mt.alleles[0], mt.alleles[1]) & (mt.consequence_category != "non_coding")),
                          n_URV_PTV = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "ptv")),
                          n_URV_PTV_LC = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "ptv_lc")),
                          n_URV_damaging_missense = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "damaging_missense")),
                          n_URV_other_missense = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "other_missense")),
                          n_URV_synonymous = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "synonymous")),
                          n_URV_non_coding = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "non_coding"))
                         )

def count_homozygous_urv_by_samples(mt):
    r'''Count up homozygous variants by samples    
    :param mt: a MatrixTable with the field "consequence_category"
    '''
    return mt.annotate_cols(n_hom_coding_URV_SNP = hl.agg.count_where(mt.GT.is_hom_var() & hl.is_snp(mt.alleles[0], mt.alleles[1]) & (mt.consequence_category != "non_coding")),
                          n_hom_coding_URV_indel = hl.agg.count_where(mt.GT.is_hom_var() & hl.is_indel(mt.alleles[0], mt.alleles[1]) & (mt.consequence_category != "non_coding")),
                          n_hom_URV_PTV = hl.agg.count_where(mt.GT.is_hom_var() & (mt.consequence_category == "ptv")),
                          n_hom_URV_PTV_LC = hl.agg.count_where(mt.GT.is_hom_var() & (mt.consequence_category == "ptv_lc")),
                          n_hom_URV_damaging_missense = hl.agg.count_where(mt.GT.is_hom_var() & (mt.consequence_category == "damaging_missense")),
                          n_hom_URV_other_missense = hl.agg.count_where(mt.GT.is_hom_var() & (mt.consequence_category == "other_missense")),
                          n_hom_URV_synonymous = hl.agg.count_where(mt.GT.is_hom_var() & (mt.consequence_category == "synonymous")),
                          n_hom_URV_non_coding = hl.agg.count_where(mt.GT.is_hom_var() & (mt.consequence_category == "non_coding"))
                         )

def count_urv_by_genes(mt):
    r''' Count up URVs by gene. Requires using the "count_urv_by_sample" function first.
    '''
    return (mt.group_rows_by(mt.consequence.vep.worst_csq_for_variant_canonical.gene_id).
            aggregate(by_gene = hl.struct(
                coding_URV_SNP = hl.agg.sum(mt.n_coding_URV_SNP),
                coding_URV_indel = hl.agg.sum(mt.n_coding_URV_indel),
                URV_PTV = hl.agg.sum(mt.n_URV_PTV),
                URV_PTV_LC = hl.agg.sum(mt.n_URV_PTV_LC),
                URV_damaging_missense = hl.agg.sum(mt.n_URV_damaging_missense),
                URV_other_missense = hl.agg.sum(mt.n_URV_other_missense),
                URV_synonymous = hl.agg.sum(mt.n_URV_synonymous),
                URV_non_coding = hl.agg.sum(mt.n_URV_non_coding)
            )      
        )
    )

2021-10-25 12:49:33 Hail: ERROR: Analysis exception: 'MatrixTable.annotate_cols': name collision with field indexed by ['row']: 'n_coding_URV_SNP'


ExpressionException: 'MatrixTable.annotate_cols': name collision with field indexed by ['row']: 'n_coding_URV_SNP'

In [65]:
mt = count_urv_by_gene(mt1)
mt.consequence.vep.worst_csq_for_variant_canonical.gene_id

<StringExpression of type str>

In [78]:
(mt.group_rows_by(mt.consequence.vep.worst_csq_for_variant_canonical.gene_id).
            aggregate(test = hl.struct(
                coding_URV_SNP = hl.agg.sum(mt.n_coding_URV_SNP),
                coding_URV_indel = hl.agg.sum(mt.n_coding_URV_indel),
                URV_PTV = hl.agg.sum(mt.n_URV_PTV),
                URV_PTV_LC = hl.agg.sum(mt.n_URV_PTV_LC),
                URV_damaging_missense = hl.agg.sum(mt.n_URV_damaging_missense),
                URV_other_missense = hl.agg.sum(mt.n_URV_other_missense),
                URV_synonymous = hl.agg.sum(mt.n_URV_synonymous),
                URV_non_coding = hl.agg.sum(mt.n_URV_non_coding)
            )      
        )
)

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
    'eur': int32
----------------------------------------
Row fields:
    'gene_id': str
----------------------------------------
Entry fields:
    'test': struct {
        coding_URV_SNP: int64, 
        coding_URV_indel: int64, 
        URV_PTV: int64, 
        URV_PTV_LC: int64, 
        URV_damaging_missense: int64, 
        URV_other_missense: int64, 
        URV_synonymous: int64, 
        URV_non_coding: int64
    }
----------------------------------------
Column key: ['s']
Row key: ['gene_id']
----------------------------------------


In [67]:
test = (mt.group_rows_by(mt.consequence.vep.worst_csq_for_variant_canonical.gene_id).
            aggregate(coding_URV_SNP = hl.agg.sum(mt.n_coding_URV_SNP)))
test.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
    'eur': int32
----------------------------------------
Row fields:
    'gene_id': str
----------------------------------------
Entry fields:
    'coding_URV_SNP': int64
----------------------------------------
Column key: ['s']
Row key: ['gene_id']
----------------------------------------


In [62]:
mt.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
    'eur': int32
----------------------------------------
Row fields:
    'gene_id': str
----------------------------------------
Entry fields:
    'coding_URV_SNP': int64
----------------------------------------
Column key: ['s']
Row key: ['gene_id']
----------------------------------------


In [34]:
mt = (mt.group_rows_by(mt.consequence.vep.worst_csq_for_variant_canonical.gene_id).
            aggregate_rows(urv_counts = hl.struct(
                coding_URV_SNP = hl.agg.sum(mt.n_coding_URV_SNP))
                          )
     )

In [35]:
mt.describe()

AttributeError: 'list' object has no attribute 'items'

In [5]:
mt1 = mt1.explode_rows(mt1.consequence.vep.worst_csq_by_gene_canonical)
by_gene_annotation1 = analysis.annotation_case_builder(mt1.consequence.vep.worst_csq_by_gene_canonical, mt1.consequence.dbnsfp, use_loftee = False)
mt1 = mt1.annotate_rows(consequence_category = by_gene_annotation1) 
mt1_subset = mt1.filter_rows(hl.literal(set(['damaging_missense'])).contains(mt1.consequence_category)) 

In [6]:
#mt1_subset.count()

In [7]:
#print(mt1_subset.count())
ht = mt1_subset
ht = ht.drop('consequence')
out = hl.vep(ht, 'utils/configs/vep_dbNSFP.json')


In [9]:
out.vep.transcript_consequences.PROVEAN_pred.show()

2021-10-24 17:00:30 Hail: WARN: Can't convert JSON value JArray(List(JString(Q6XZB0))) to type str at <root>.transcript_consequences[element].swissprot.
2021-10-24 17:00:30 Hail: WARN: Can't convert JSON value JArray(List(JString(UPI000462096D))) to type str at <root>.transcript_consequences[element].uniparc.
2021-10-24 17:00:30 Hail: WARN: struct{allele_num: int32, amino_acids: str, biotype: str, canonical: int32, ccds: str, cdna_start: int32, cdna_end: int32, cds_end: int32, cds_start: int32, codons: str, consequence_terms: array<str>, distance: int32, domains: array<struct{db: str, name: str}>, exon: str, gene_id: str, gene_pheno: int32, gene_symbol: str, gene_symbol_source: str, hgnc_id: str, hgvsc: str, hgvsp: str, hgvs_offset: int32, impact: str, intron: str, lof: str, lof_flags: str, lof_filter: str, lof_info: str, minimised: int32, polyphen_prediction: str, polyphen_score: float64, protein_end: int32, protein_start: int32, protein_id: str, sift_prediction: str, sift_score: floa

2021-10-24 17:00:31 Hail: WARN: struct{aa_allele: str, aa_maf: float64, afr_allele: str, afr_maf: float64, allele_string: str, amr_allele: str, amr_maf: float64, clin_sig: array<str>, end: int32, eas_allele: str, eas_maf: float64, ea_allele: str, ea_maf: float64, eur_allele: str, eur_maf: float64, exac_adj_allele: str, exac_adj_maf: float64, exac_allele: str, exac_afr_allele: str, exac_afr_maf: float64, exac_amr_allele: str, exac_amr_maf: float64, exac_eas_allele: str, exac_eas_maf: float64, exac_fin_allele: str, exac_fin_maf: float64, exac_maf: float64, exac_nfe_allele: str, exac_nfe_maf: float64, exac_oth_allele: str, exac_oth_maf: float64, exac_sas_allele: str, exac_sas_maf: float64, id: str, minor_allele: str, minor_allele_freq: float64, phenotype_or_disease: int32, pubmed: array<int32>, sas_allele: str, sas_maf: float64, somatic: int32, start: int32, strand: int32} has no field AA at <root>.colocated_variants[element] for value JString(A:0)
2021-10-24 17:00:31 Hail: WARN: struct{a

,,
locus,alleles,
locus<GRCh38>,array<str>,array<str>
chr21:14165259,"[""A"",""T""]","[NA,NA,NA,NA]"
chr21:14165285,"[""T"",""A""]","[NA,NA,NA,NA]"
chr21:14165372,"[""C"",""T""]","[NA,NA,NA,NA]"
chr21:14166382,"[""C"",""G""]","[NA,NA,NA,NA]"
chr21:14166389,"[""G"",""T""]","[NA,NA,NA,NA]"
chr21:14166421,"[""A"",""G""]","[NA,NA,NA,NA]"
chr21:14166421,"[""A"",""T""]","[NA,NA,NA,NA]"
chr21:14181778,"[""A"",""G""]","[NA,NA,NA,NA]"


In [37]:
def gene_csqs_case_builder(in_mt):
    r''' Returns matrix table that contains gene consequence information from phased geneotypes.
    "": no alternate alleles,
    "HE": one alternate allele on either strand in a locus, 
    "HO": homozygous for alternate alleles
    "CH": two alternate allele on either strand in a locus (compound heterozygous)
    "CH+HO": two alternate allele on either strand in a locus (either as homozygous or compound heterozygous)
    '''
    # create one gene_id for each item in gene_id array
    #in_mt = in_mt.explode_rows(in_mt.vep.worst_csq_by_gene_canonical)
    # get all snps that are not homozygous
    mt = in_mt
    mt = analysis.annotate_phased_entries(mt)
    mt = mt.filter_entries(~mt.GT.is_hom_var())
    # create table for each strand and combine to gene
    ht0 = (mt.group_rows_by(mt.consequence.vep.worst_csq_by_gene_canonical.gene_id).aggregate(s0 = hl.agg.any(mt.a0_alt)))
    ht1 = (mt.group_rows_by(mt.consequence.vep.worst_csq_by_gene_canonical.gene_id).aggregate(s1 = hl.agg.any(mt.a1_alt)))
    ht2 = (in_mt.group_rows_by(in_mt.consequence.vep.worst_csq_by_gene_canonical.gene_id).aggregate(hom_var = hl.agg.any(in_mt.GT.is_hom_var())))
    # combine entries
    ht = ht0.annotate_entries(s1 = ht1[ht0.gene_id, ht0.s].s1)
    ht = ht.annotate_entries(hom_var = ht2[ht.gene_id, ht.s].hom_var)
    expr = (hl.case()
           .when( (ht.s0) & (ht.s1) & (ht.hom_var), 'CH+HO')
           .when( (ht.s0) & (ht.s1), "CH")
           .when( (ht.hom_var), 'HO')
           .when( (ht.s0) & (ht.s1 == False), 'HE')
           .when( (ht.s1) & (ht.s0 == False), 'HE')
           .default(''))
    ht = ht.annotate_entries(csqs = expr)
    ht = ht.drop('s0').drop('s1').drop('hom_var')    
    return ht

In [63]:
mt.consequence.vep.worst_csq_by_gene_canonical.gene_id.dtype

dtype('array<str>')

In [67]:
mt.consequence.vep.worst_csq_by_gene_canonical.gene_id.dtype == hl.expr.ArrayExpression

False

In [71]:
hl.expr.ArrayExpression(hl.str(''))

TypeError: __init__() missing 1 required positional argument: 'type'

TypeError: __init__() missing 2 required positional arguments: 'x' and 'type'